# Alternative RCM BA - 1. Reference quantiles at 0.25 deg (F1 + BCSD)

**Alternative solution** (to the supervisor's downscaling pipeline) for the bias adjustment of the RCM:

1. **F1** (`GCM_BA/`, done): WDF + QDM of EC-Earth3 towards ERA5 on the EC-Earth3 grid (0.7 deg), applied to
   `gcm_2025_nohosing` -> the target climate, ERA5-like, at 0.7 deg.
2. **Reference quantiles at 0.25 deg** (BCSD principle, Wood et al. 2004), per calendar month m (the 3 months centred on
   m, as F1) and quantile: Q_ref = Q_ERA5,0.25 + I[Q_F1 - Q_ERA5,0.7] for tasmax, ratios (interpolated in log) for DTR
   and the pr wet days; pr: dry-day fraction first, p0_ref = p0_ERA5,0.25 + I[p0_F1 - p0_ERA5,0.7] (ISIMIP3BASD
   principle, Lange 2019). I = bilinear interpolation from 0.7 to 0.25 deg. The RCM does not enter the reference.
3. **F2**: QDM of `rcm_2025_nohosing` towards Q_ref (`bu.fit_F2`: the xsdba training with Q_ref as ref quantiles),
   applied unchanged to the four RCM runs with `bu.apply_F` (the code of F1), each run ranked in its own distribution.

The RCM keeps its own day-to-day sequence and spatial structure (they carry the AMOC effect): the correction acts on
the distribution of each pixel only.

This notebook only needs ERA5 (0.25 deg and on the EC-Earth3 grid) and the F1 product of `gcm_2025_nohosing`: it runs
before the RCM is available. The reference then covers all the years of the F1 product; once the RCM is in the config,
it is recomputed over the years of `rcm_2025_nohosing` (`REF_YEARS`, in the file name).

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                       # pr, tasmax, tasmin (tasmin through DTR, as F1)

# ---- F1 (GCM_BA): the same tags as GCM_BA notebooks 2-3
CAL_PERIOD = ("1981", "2014")                 # calibration period of F1 (ERA5 / EC-Earth3 historical)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}_europe"
F1_PRODUCTS = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")

# ---- alternative solution
TARGET_GCM = "gcm_2025_nohosing"              # F1 product that gives the target climate of the RCM calibration
RCM_CAL = "rcm_2025_nohosing"                 # RCM run driven by it: F2 calibrated on it
RCM_RUNS = ["rcm_2025_nohosing", "rcm_2025_hosing", "rcm_2050_nohosing", "rcm_2050_hosing"]
MARGIN = 1.0                                  # degrees around the EFAS domain (LISFLOOD Europe)
REGION = {k: (bu.EFAS_BOX[k][0] - MARGIN, bu.EFAS_BOX[k][1] + MARGIN) for k in ("lat", "lon")}
COARSE_REGION = {k: (REGION[k][0] - 1.0, REGION[k][1] + 1.0) for k in ("lat", "lon")}   # + 1 coarse cell, for the interpolation
CHUNKS_SPACE = {"lat": 32, "lon": 64}
RCM_READY = bu.dataset_available(cfg, RCM_CAL, "pr")


def ref_years():
    """Years of the reference = years of the RCM calibration run (the F1 product over the same years); as long as the
    RCM is not in the config: all the years of the F1 product (to recompute once the RCM arrives)."""
    src = bu.read_raw(cfg, RCM_CAL, "pr") if RCM_READY else bu.read_processed(F1_PRODUCTS / f"{TARGET_GCM}_{F1_TAG}.nc", "pr")
    y = src.time.dt.year.values
    return str(y.min()), str(y.max())


REF_YEARS = ref_years()
QREF_TAG = f"{F1_TAG}_{TARGET_GCM}_{REF_YEARS[0]}-{REF_YEARS[1]}"
QREF_FILE = bu.alt_dir(cfg, "1_reference_quantiles") / f"qref_{QREF_TAG}.nc"
F2_DIR = bu.alt_dir(cfg, "2_F2_calculation", f"F2_{QREF_TAG}")
OUT = bu.alt_dir(cfg, "RCM_bias_adjusted_0.25deg")
print(f"region {REGION} | reference years {REF_YEARS[0]}-{REF_YEARS[1]}"
      + ("" if RCM_READY else " (RCM not in the config yet: all the years of the F1 product)"))
print("reference quantiles:", QREF_FILE, "\nF2:", F2_DIR, "\nproducts:", OUT)

## 1. Inputs

ERA5 at 0.25 deg over the region (EFAS domain + 1 deg) and `CAL_PERIOD`: extracted once from the global daily files
(`0_inputs/`, chunks of 31 days), since the quantiles of the 12 windows read every day three times. ERA5 on the EC-Earth3
grid and the F1 product: as in `GCM_BA` (over `COARSE_REGION`).

In [ ]:
E_FILE = bu.alt_dir(cfg, "0_inputs") / f"era5_025_efas{MARGIN:g}_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}.nc"
if not E_FILE.exists():
    t0 = time.time()
    e = {v: bu.subset_time(bu.crop_bbox(bu.read_raw(cfg, "era5", v), REGION), *CAL_PERIOD) for v in VARS}
    bu.save_ds(xr.Dataset(e), E_FILE)
    print(f"ERA5 0.25 deg extracted in {(time.time() - t0) / 60:.1f} min")
era5_fine = {v: bu.read_processed(E_FILE, v) for v in VARS}
era5_coarse = {v: bu.gcm_ba_input(cfg, "era5", v, CAL_PERIOD, COARSE_REGION) for v in VARS}
f1 = {v: bu.subset_time(bu.crop_bbox(bu.read_processed(F1_PRODUCTS / f"{TARGET_GCM}_{F1_TAG}.nc", v), COARSE_REGION), *REF_YEARS)
      for v in VARS}
for lab, d in (("ERA5 0.25 deg", era5_fine), ("ERA5 0.7 deg", era5_coarse), (f"F1 {TARGET_GCM}", f1)):
    x = d["pr"]
    print(f"{lab:22s} {x.sizes['lat']} x {x.sizes['lon']} pixels, {x.sizes['time']} days "
          f"({x.time.dt.year.values.min()}-{x.time.dt.year.values.max()})")

## 2. Reference quantiles (`bu.reference_quantiles`)

In [ ]:
OVERWRITE = False
if QREF_FILE.exists() and not OVERWRITE:
    print("[already done]", QREF_FILE.name)
else:
    t0 = time.time()
    qref = bu.reference_quantiles(era5_fine, era5_coarse, f1, chunks_space=CHUNKS_SPACE)
    qref.attrs.update({"f1_tag": F1_TAG, "target_gcm": TARGET_GCM, "ref_years": "-".join(REF_YEARS),
                       "era5_period": "-".join(CAL_PERIOD), "region": json.dumps(REGION)})
    qref.to_netcdf(QREF_FILE)
    print(f"reference quantiles in {(time.time() - t0) / 60:.1f} min")
qref = xr.open_dataset(QREF_FILE).load()
qref

## 3. Checks

The change the reference adds to ERA5 at 0.25 deg (Q_ref - Q_ERA5,0.25, in % for pr) for January and July, at the
median and at the 90th percentile; the dry-day fraction of ERA5 and of the reference. Smooth fields (interpolated from
0.7 deg), with the fine-scale structure of ERA5 underneath.

In [ ]:
q_e = {"tasmax": bu.window_quantiles(era5_fine["tasmax"], int(qref.attrs["nquantiles"]), chunks_space=CHUNKS_SPACE).compute()}
p0_e = bu.window_dry_fraction(era5_fine["pr"]).compute()
land = bu.keep_mask(qref) & bu.load_domain_mask(cfg, "era5", like=qref.q_tasmax.isel(month=0, quantiles=0))
fig, axes = plt.subplots(2, 4, figsize=(22, 8))
for r, m in enumerate([1, 7]):
    for c, tau in enumerate([0.5, 0.9]):
        d = (qref.q_tasmax - q_e["tasmax"]).sel(month=m).sel(quantiles=tau, method="nearest").where(land)
        d.plot(ax=axes[r, c], cmap="RdBu_r", center=0, cbar_kwargs={"shrink": 0.7, "label": "degC"})
        axes[r, c].set_title(f"tasmax p{int(100 * tau)}, month {m}: Q_ref - Q_ERA5,0.25")
    for c, (lab, a) in enumerate((("ERA5 0.25", p0_e), ("reference", qref.p0_pr)), start=2):
        a.sel(month=m).where(land).plot(ax=axes[r, c], vmin=0, vmax=1, cmap="YlGnBu_r", cbar_kwargs={"shrink": 0.7})
        axes[r, c].set_title(f"pr dry-day fraction, month {m}: {lab}")
for ax in axes.ravel():
    ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.savefig(QREF_FILE.with_suffix(".png"), dpi=90); plt.show()
rows = [{"month": m, "tasmax p50 change (degC)": float((qref.q_tasmax - q_e["tasmax"]).sel(month=m).sel(quantiles=0.5, method="nearest").where(land).mean()),
         "dry-day fraction ERA5": float(p0_e.sel(month=m).where(land).mean()), "dry-day fraction ref": float(qref.p0_pr.sel(month=m).where(land).mean())}
        for m in range(1, 13)]
display(pd.DataFrame(rows).set_index("month").round(3))